# 🚀 LLM in Python - From Simple to Complex

This notebook walks through practical Groq API examples, progressively building in complexity.

| # | Example | Difficulty |
|---|---------|------------|
| 1 | Setup & Basic Chat | ⭐ |
| 2 | System Prompts & Personas | ⭐⭐ |
| 3 | Streaming Responses | ⭐⭐ |
| 6 | Sentiment Analysis Pipeline | ⭐⭐⭐ |

**> Get your free API key at:** https://console.groq.com/keys

## 🔧 Setup — Install & Configure

In [ ]:
!pip install groq faiss-cpu sentence-transformers ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.4 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

# Option A: Store your key in Colab Secrets (recommended)
# Go to the key icon in the left sidebar and add GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get('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')
    print('Loaded API key from Colab Secrets')
except Exception:
    GROQ_API_KEY = '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'  # Replace with your key
    print('Using hardcoded key - prefer Colab Secrets for security')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

Using hardcoded key - prefer Colab Secrets for security


In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODELS = {
    'fast':     'llama-3.1-8b-instant',
    'balanced': 'llama-3.3-70b-versatile',
    'code':     'qwen-2.5-coder-32b',
}

print('Groq client ready!')
print('Models:', list(MODELS.keys()))

Groq client ready!
Models: ['fast', 'balanced', 'code']


---
## Example 1 - Basic Chat Completion

In [ ]:
response = client.chat.completions.create(
    model=MODELS['fast'],
    messages=[
        {"role": "user", "content": "What is Groq and why is it so fast?"}
    ]
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")
print(f"Speed: {response.usage.completion_tokens / response.usage.completion_time:.0f} tokens/sec")

Groq is a US-based company founded in 2018 that specializes in developing high-performance artificial intelligence (AI) and deep neural network (DNN) computing hardware. Their product, the Groq Chip, is a highly optimized chip designed specifically for AI and DNN workloads.

The Groq Chip is built using a combination of software and hardware innovations that enable it to achieve exceptional performance and efficiency. Here are some key factors that contribute to its speed:

1. **Architecture:** The Groq Chip uses a novel architecture that is designed to minimize latency and maximize throughput. It features a hierarchical matrix multiplication engine that can perform multiple operations in parallel, reducing the number of memory accesses and computational cycles required for matrix multiplication, which is a fundamental operation in AI and DNN computing.
2. **Memory Architecture:** The chip features a high-bandwidth, low-latency on-chip memory system that allows for fast access to data.

---
## Example 2 - System Prompts & Personas

In [ ]:
def chat_with_persona(persona_prompt, user_message, model=None):
    model = model or MODELS['fast']
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": persona_prompt},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.7,
        max_tokens=512
    )
    return response.choices[0].message.content


socratic = """
You are a Socratic teacher. Never give direct answers.
Guide the student with thoughtful questions only. Keep responses under 80 words.
"""
print("Socratic Teacher:")
print(chat_with_persona(socratic, "Why does the sky appear blue?"))

print("\n" + "-"*60 + "\n")

pirate_chef = """
You are a salty old pirate who is also a Michelin-star chef.
Speak in pirate dialect but give genuine, expert cooking advice.
"""
print("Pirate Chef:")
print(chat_with_persona(pirate_chef, "How do I make the perfect risotto?"))

Socratic Teacher:
What is your understanding of the colors we see in the world around us? How do you think light behaves when it interacts with the objects it encounters?

------------------------------------------------------------

Pirate Chef:
Arrr, ye landlubber! Ye be wantin' to know the secrets o' the perfect risotto, eh? Alright then, listen close and I'll share me expertise with ye. But first, find yerself a good bottle o' wine, 'cause we're gonna get cookin' like proper scurvy dogs!

**Gather yer ingredients, matey:**

- 1 cup Arborio rice (the good stuff, not that scrawny, long-grain rice)
- 4 cups vegetable or chicken broth, warmed (not too hot, not too cold, just like me temper)
- 2 tablespoons olive oil (the good, extra-virgin kind, not that watered-down bilge)
- 1 small onion, finely chopped (no tears, matey, just a smooth, even chop)
- 2 cloves garlic, minced (vampires be warned: garlic be keepin' them at bay)
- 1 cup white wine (a good dry white, like a crisp sea breeze

---
## Example 3 - Sentiment Analysis Pipeline

Process a batch of customer reviews, classify each with fine-grained sentiment, extract topics, and produce a summary report.

In [ ]:
import json
from collections import Counter

reviews = [
    "Absolutely love this product! Fast shipping and great quality. Will buy again.",
    "Terrible experience. Item arrived broken and customer service was unhelpful.",
    "It's okay. Does what it's supposed to do but nothing special.",
    "Amazing quality for the price! Blew my expectations out of the water.",
    "Returned it after 2 days. Poor build quality and the size was wrong.",
    "Decent product but delivery took 3 weeks longer than expected.",
    "Five stars! The packaging was beautiful and the item works perfectly.",
    "Not bad, not great. Instructions were confusing but it works fine now.",
]


def analyze_sentiment(review):
    response = client.chat.completions.create(
        model=MODELS['fast'],
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a sentiment analysis engine. "
                    "Return ONLY a JSON object with these exact keys:\n"
                    '  "sentiment": one of ["very_positive", "positive", "neutral", "negative", "very_negative"]\n'
                    '  "score": float from -1.0 (most negative) to 1.0 (most positive)\n'
                    '  "topics": list of mentioned topics (e.g. ["shipping", "quality", "price"])\n'
                    '  "emotion": dominant emotion (e.g. "joy", "anger", "disappointment", "satisfaction")\n'
                    '  "one_line": a single sentence summary of the review'
                )
            },
            {"role": "user", "content": review}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )
    result = json.loads(response.choices[0].message.content)
    result["review"] = review
    return result


print("Analyzing reviews...\n")
results = [analyze_sentiment(r) for r in reviews]

EMOJI = {
    "very_positive": "😍",
    "positive":      "😊",
    "neutral":       "😐",
    "negative":      "😞",
    "very_negative": "😡",
}

for i, r in enumerate(results, 1):
    emoji = EMOJI.get(r['sentiment'], '?')
    bar_len = int((r['score'] + 1) / 2 * 20)
    bar = '#' * bar_len + '.' * (20 - bar_len)
    print(f"Review {i}: {emoji} {r['sentiment']:15s} [{bar}] score={r['score']:+.2f}")
    print(f"  Topics : {', '.join(r['topics'])}")
    print(f"  Emotion: {r['emotion']}")
    print(f"  Summary: {r['one_line']}")
    print()

print("=" * 60)
print("AGGREGATE REPORT")
print("=" * 60)

avg_score = sum(r['score'] for r in results) / len(results)
sentiment_counts = Counter(r['sentiment'] for r in results)
all_topics = [t for r in results for t in r['topics']]
top_topics = Counter(all_topics).most_common(5)

print(f"Average sentiment score : {avg_score:+.2f}")
print(f"Sentiment distribution  : {dict(sentiment_counts)}")
print(f"Top mentioned topics    : {top_topics}")

positive_pct = (sentiment_counts.get('very_positive', 0) +
                sentiment_counts.get('positive', 0)) / len(results) * 100
print(f"Positive reviews        : {positive_pct:.0f}%")

Analyzing reviews...

Review 1: 😍 very_positive   [###################.] score=+0.93
  Topics : shipping, quality
  Emotion: joy
  Summary: Excellent product with fast shipping and great quality.

Review 2: 😡 very_negative   [....................] score=-0.93
  Topics : customer service, shipping
  Emotion: anger
  Summary: The customer service was unhelpful and the item arrived broken.

Review 3: 😐 neutral         [##########..........] score=+0.00
  Topics : performance
  Emotion: neutrality
  Summary: It meets expectations.

Review 4: 😍 very_positive   [###################.] score=+0.95
  Topics : quality, price
  Emotion: joy
  Summary: Exceeded expectations with great quality at a reasonable price.

Review 5: 😡 very_negative   [#...................] score=-0.80
  Topics : build quality, size
  Emotion: anger
  Summary: The product has poor build quality and incorrect size.

Review 6: 😐 neutral         [############........] score=+0.20
  Topics : shipping, product
  Emotion: frust

---
## Quick Reference

| Concept | Key param / method |
|---|---|
| Pick a model | `model=` |
| Control creativity | `temperature=` (0 = deterministic, 1 = creative) |
| Limit output | `max_tokens=` |
| Force JSON output | `response_format={"type": "json_object"}` |
| Stream tokens | `client.chat.completions.stream(...)` |
| Async / concurrent | `AsyncGroq` + `asyncio.gather(...)` |
| Use tools | `tools=[...]`, `tool_choice="auto"` |
| OpenAI compat | `base_url="https://api.groq.com/openai/v1"` |

**Useful links:**
- [Groq Docs](https://console.groq.com/docs)
- [API Keys](https://console.groq.com/keys)
- [Model List](https://console.groq.com/docs/models)
- [Discord Community](https://discord.gg/groq)